# 01 | Carga y auditoría de datos

> **Cuaderno de reproducción (posterior).** Generado el 2026-10-23 como parte de la entrega docente. El repositorio original no contenía cuadernos versionados; estos reconstruyen los análisis a partir de los scripts y módulos existentes.

**Tipo de ejecución: CARGA DE DATOS — lee los CSV crudos; NO entrena modelos.**

In [1]:
import os
from pathlib import Path

def _raiz_repositorio() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'interseccion_terrenos_zhg_r1_yacopi.csv').exists():
            return p
    raise FileNotFoundError('No se encontro la raiz del repositorio (busca data/raw)')

REPO = _raiz_repositorio()
print('Raiz del repositorio resuelta (carpeta):', REPO.name, '| data/raw encontrado')

Raiz del repositorio resuelta (carpeta): yacopi-project | data/raw encontrado


## Fuentes

Carga de los dos CSV crudos y resumen de calidad.

In [2]:
import pandas as pd
inter = pd.read_csv(REPO / 'data/raw/interseccion_terrenos_zhg_r1_yacopi.csv', low_memory=False)
zhf = pd.read_csv(REPO / 'data/raw/zhf_yacopi.csv', low_memory=False)
print('intersecciones:', inter.shape, '| predios unicos:', inter.numero_predial.nunique())
print('zhf:', zhf.shape, '| codpred unicos:', zhf.CODPRED.nunique())
print('sha256 interseccion:', '8402f527a5468f94a80753b547f54c64ee96e4a938bc2afe4d119b488a3d6875')
print('sha256 zhf:', '9bd88923146530ade9ba9eb964d8e97aee32e28ce56e4b95b08e380b684e15ad')

intersecciones: (9447, 17) | predios unicos: 7709
zhf: (7968, 28) | codpred unicos: 7921
sha256 interseccion: 8402f527a5468f94a80753b547f54c64ee96e4a938bc2afe4d119b488a3d6875
sha256 zhf: 9bd88923146530ade9ba9eb964d8e97aee32e28ce56e4b95b08e380b684e15ad


In [3]:
cols = ['valor_avaluo','area_terreno_m2','area_construida_m2']
resumen = inter[cols].agg(['count', lambda s: s.isna().sum(), lambda s: (s==0).sum(), lambda s: (s<0).sum(), 'min','median','max'])
resumen.index = ['n','nulos','ceros','negativos','min','mediana','max']
resumen

,valor_avaluo,area_terreno_m2,area_construida_m2
n,9447.0,9447.0,9447.0
nulos,0.0,0.0,0.0
ceros,0.0,0.0,5793.0
negativos,0.0,0.0,0.0
min,4000.0,10.0,0.0
mediana,2818000.0,50000.0,0.0
max,878808000.0,8306726.0,2016.0


## Lectura

Sin nulos ni negativos en avalúo y áreas; 5.793 filas con construcción = 0. La construcción faltante NO se interpreta como ausencia comprobada de edificaciones.